# Rung 30b — score one checkpoint of one arm

One notebook run = **one arm × one checkpoint**. Driven by papermill:
`-p SMOKE False -p RUN 30_grpo_v1_grpo_full -p STEP 600`.

## The pre-registration is enforced HERE, in code, not in prose

`README.md` §"Which checkpoint carries the verdict" declares:

1. the verdict pair is **`checkpoint-600` vs `checkpoint-600`** — step-matching is the whole design;
2. the five intermediates are the **collapse guard only, never selection**;
3. a 600 with significant harm on `object_recognition` is **NO-GO**, with no fallback to a
   surviving earlier checkpoint.

So this notebook routes its output by step: **`STEP == 600` writes `RESULTS_verdict.csv`; any
other step can only write `RESULTS_collapse_guard.csv`.** An intermediate checkpoint is
physically unable to land in the file the verdict is read from. That is the difference between a
declared rule and an enforced one, and this rung has already paid once for the difference
(see the control that trained on nothing, `f5aec95`).

## What it does not do

No adjudication. This writes rows; the S8 clauses are read against them afterwards, once both
arms exist. A notebook that scored and judged in the same pass could not be re-run on one arm.

In [ ]:
# --- bootstrap -----------------------------------------------------------------
import json, os, shutil, sys, time
from pathlib import Path
import pandas as pd

# `merge_checkpoint` shells out to the bare `swift` binary. A papermill kernel does NOT
# inherit the env's bin/ on PATH, and it must be THIS interpreter's bin.
_envbin = str(Path(sys.executable).parent)
if _envbin not in os.environ.get("PATH", "").split(os.pathsep):
    os.environ["PATH"] = _envbin + os.pathsep + os.environ.get("PATH", "")

EXP = Path.cwd()
REPO = EXP
while REPO != REPO.parent and not (REPO / "src").is_dir():
    REPO = REPO.parent
if EXP.name != "30-grpo-number":
    EXP = REPO / "experiments" / "30-grpo-number"
for p in (REPO / "src", REPO / "vendor" / "orena-focus" / "src",
          EXP / "_models",
          REPO / "experiments" / "21-recipe-sweep" / "_models",
          REPO / "experiments" / "02-lora-sft" / "_models"):
    if p.is_dir() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

from frame import ledger, metrics
from frame.config import BaselineConfig
from frame.run import run_baseline
from recipe_sweep_train import RecipeSweepConfig, list_checkpoints, merge_checkpoint
print("repo:", REPO)

In [ ]:
# --- parameters (RAW LITERALS ONLY — papermill injects BELOW this cell) ----------
SMOKE = True                      # True -> 40 questions, wiring only. Full: -p SMOKE False
RUN   = "30_grpo_v1_grpo_full"    # or 30_grpo_v1_control_full
STEP  = 600                       # 100..600. Only 600 may write the verdict file.

KEEP_MERGED = False               # a merged checkpoint is ~17 GB; the volume has ~90 GB free
DATA_ROOT   = "/workspace/orena-data"

In [ ]:
# --- derived (MUST live BELOW the parameters cell — the rung-16 papermill trap) --
ARM = "grpo" if "_grpo_" in RUN else "control"
RUN_DIR = EXP / "runs" / RUN
RUN_TAG = f"step{STEP}_{'smoke' if SMOKE else 'full'}"

# The QA parquets live in different places on the pod and on a laptop. Resolve by LOOKING,
# and fail loudly: an eval on an empty data root is the classic silent zero.
DATA_ROOT = next(
    (d for d in (Path(DATA_ROOT), REPO / "external_data" / "orena-data")
     if (d / "heico" / "data" / "frame" / "test.parquet").exists()), None)
assert DATA_ROOT is not None, "no frame/test.parquet found — pull the QA parquets"

# rung 30's run layout is rung 21's layout: runs/<RUN>/ckpt and runs/<RUN>/merged. No new
# engine is needed; the checkpoints are STEP-numbered rather than epoch-numbered, which
# list_checkpoints already handles (it sorts on the trailing integer).
cfg = RecipeSweepConfig(exp_dir=EXP, run_name=RUN, data_root=DATA_ROOT)
CKPTS = list_checkpoints(cfg)
CKPT = next((c for c in CKPTS if c.name == f"checkpoint-{STEP}"), None)
assert CKPT is not None, (
    f"no checkpoint-{STEP} under {cfg.ckpt_dir}; found {[c.name for c in CKPTS]}")

# 🔴 THE PRE-REGISTRATION, ENFORCED. README §"Which checkpoint carries the verdict".
# The intermediates are the collapse guard and may never reach the file the verdict is read
# from. Routing this in code rather than in discipline is the point: with six checkpoints per
# arm, a human choosing where to write the row IS the six degrees of freedom the
# pre-registration exists to remove.
VERDICT_STEP = 600
OUT_CSV = EXP / ("RESULTS_verdict.csv" if STEP == VERDICT_STEP
                 else "RESULTS_collapse_guard.csv")
print(f"arm={ARM} run={RUN} ckpt={CKPT.name} smoke={SMOKE}")
print(f"row -> {OUT_CSV.name}"
      + ("  (VERDICT)" if STEP == VERDICT_STEP else "  (collapse guard, never selection)"))

In [ ]:
# --- PRE-FLIGHT: the LLM judge must resolve offline, BEFORE anything expensive ---
# `run_baseline` loads the judge only AFTER the 17 GB merge and the full inference pass, so a
# missing judge cache fails ~45 minutes in with everything already paid for. This is that
# failure, hoisted to the front and made cheap: the tokenizer alone proves the cache resolves
# under the offline flags. RAISES (RULES §7).
from transformers import AutoTokenizer

_judge = BaselineConfig().judge_model
AutoTokenizer.from_pretrained(_judge)
print("judge resolves offline:", _judge)

In [ ]:
# --- merge the adapter (per-step, namespaced so merges never overwrite) ----------
merged = cfg.merged_dir / CKPT.name
if merged.is_dir() and any(merged.iterdir()):
    print(f"OK    merged already present -> {merged}")
    MERGED_HERE = False
else:
    t0 = time.perf_counter()
    merged = merge_checkpoint(cfg, CKPT)
    MERGED_HERE = True
    print(f"      merged in {time.perf_counter() - t0:.0f}s -> {merged}")
merged

In [ ]:
# --- eval -----------------------------------------------------------------------
t0 = time.perf_counter()
try:
    cfg_eval = BaselineConfig(
        data_root=DATA_ROOT, model_path=merged, out_dir=RUN_DIR, run_name=RUN_TAG,
        max_pixels=1280 * 720, seed=42, n_eval=40 if SMOKE else None,
    )
    # The inference path must stay rung 21's EXACTLY. This rung's variable is the training
    # OBJECTIVE (RL vs step-matched SFT); a post-processor or a second sample here would be a
    # second variable and the delta would stop being attributable to the objective alone.
    # 🔴 n_samples == 1 is load-bearing beyond the usual reason: the whole premise of this rung
    # is that pass@8 already finds the answer (0.945) and greedy does not emit it (0.705).
    # Sampling here would score the defect the arm is trying to fix.
    assert cfg_eval.answer_postprocess is None, "answer_postprocess must stay None"
    assert cfg_eval.n_samples == 1 and cfg_eval.enhance is None and cfg_eval.aux_view is None
    report = run_baseline(cfg_eval)
    print(f"eval done in {(time.perf_counter() - t0) / 60:.1f} min")
finally:
    if MERGED_HERE and not KEEP_MERGED and Path(merged).is_dir():
        shutil.rmtree(merged, ignore_errors=True)
        print(f"reclaimed ~17 GB -> removed {merged}")

In [ ]:
# --- canonical scoring + gates (all RAISE) --------------------------------------
# RULES EVAL: score ONLY via frame.metrics, leaf->group via Capability.group, ID/OOD from
# qID, headline bucket_mean. Never re-derive a metric inline in a notebook.
gold = ledger.gold_from_frame_parquets(DATA_ROOT)
res = pd.read_csv(RUN_DIR / RUN_TAG / "results.csv")

missing = set(res["qID"]) - set(gold.dropna(subset=["answer"])["qID"])
assert not missing, f"GATE 0 — {len(missing)} qIDs without gold; every margin would be inflated"

metrics.assert_no_dup_qid(res)
metrics.assert_ood_from_qid(res)
metrics.assert_all_rows_grouped(res)
strat = metrics.stratified_report(res, gold=gold)
metrics.assert_floors_vs_eval_set(strat)

# The mode gate, on the ARTIFACT rather than the variable: `-p SMOKE False` failing to take
# effect is otherwise indistinguishable from a successful full run.
assert (len(res) < 1000) == SMOKE, (
    f"MODE GATE FAILED: SMOKE={SMOKE} but the eval scored {len(res)} rows")
if not SMOKE:
    assert len(res) == 6252, f"expected the full eval set, got {len(res)} rows"

print("gates OK | bucket_mean:", round(strat["bucket_mean"], 4))

In [ ]:
# --- the cells the pre-registration named, pulled out by name --------------------
# Primary: aggregation_ID. NOT local bucket_mean — RULES §S3 forbids it as the comparator
# (it overstates the judge by +0.121 and inverts the bucket ordering). bucket_mean is carried
# anyway, as reference, because RULES §4b makes it the platform headline; the two are recorded
# side by side and must never be quoted against each other.
# Veto: object_recognition_{ID,OOD}. fo_class is 71% of object_recognition and a `number`-only
# objective is free to walk the policy away from it — the declared modal failure of this rung,
# and with beta=0 there is no KL anchor, so this column IS the collapse guard.
bb = pd.DataFrame(strat["by_bucket"] if isinstance(strat.get("by_bucket"), list)
                  else report["by_bucket"])

def cell(group: str, dist: str) -> float:
    m = bb[(bb.capability_group == group) & (bb.distribution == dist)]
    # 🔴 A missing bucket is expected in SMOKE (40 rows need not contain any aggregation×ID
    # question) and is a FINDING in a full run — NaN here, and the gate below raises only when
    # the whole eval set was scored.
    return float(m["accuracy"].iloc[0]) if len(m) else float("nan")

row = {
    "arm": ARM, "run": RUN, "step": STEP, "checkpoint": CKPT.name, "smoke": SMOKE,
    "n_scored": len(res),
    "aggregation_ID": cell("aggregation", "ID"),          # primary
    "aggregation_OOD": cell("aggregation", "OOD"),        # S8 clause 2
    "object_recognition_ID": cell("object_recognition", "ID"),   # veto
    "object_recognition_OOD": cell("object_recognition", "OOD"), # veto
    "bucket_mean": float(strat["bucket_mean"]),           # reference only
    "judge_model": BaselineConfig().judge_model,
}
if not SMOKE:
    nan_cells = [k for k in ("aggregation_ID", "aggregation_OOD",
                             "object_recognition_ID", "object_recognition_OOD")
                 if pd.isna(row[k])]
    assert not nan_cells, (
        f"declared cells missing from a FULL eval: {nan_cells} — a pre-registered cell that "
        "does not exist is not a smaller number, it is no number at all")
row

In [ ]:
# --- append the row, idempotently ------------------------------------------------
# Keyed on (run, step, smoke) so a re-run REPLACES its own row instead of stacking a second
# one. Two rows for the same checkpoint is how a ladder starts disagreeing with itself.
new = pd.DataFrame([row])
if OUT_CSV.exists():
    old = pd.read_csv(OUT_CSV)
    key = ["run", "step", "smoke"]
    old = old.merge(new[key], on=key, how="left", indicator=True)
    old = old[old["_merge"] == "left_only"].drop(columns="_merge")
    new = pd.concat([old, new], ignore_index=True)
new = new.sort_values(["arm", "step"]).reset_index(drop=True)
new.to_csv(OUT_CSV, index=False)
print(f"wrote {OUT_CSV.name}  ({len(new)} rows)")
new